In [1]:
import sys
sys.executable

'/workspaces/LLM_workspace/04_evaluation/.venv/bin/python'

---

# Module 4 Homework: Evaluation

Link to the github repo: [Link](https://github.com/DataTalksClub/llm-zoomcamp/tree/main)

## Data Loading

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [3]:
def find_doc(documents: list, filename: str):
    for doc in documents:
        if doc["filename"] == filename:
            return doc
    return None

In [4]:
find_doc(documents, "01-agentic-rag/lessons/01-intro.md")['content']

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

## Generating ground truth

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

class Questions(BaseModel):
    questions: list[str]

load_dotenv()
openai_client = OpenAI()


### Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `response.usage.input_tokens` / `prompt_tokens`).


In [7]:
from evaluation_utils import llm_structured
import json
docs_file_names = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]
docs = [{'id': file_name, 'content': find_doc(documents, file_name)['content']} for file_name in docs_file_names]
records = []

for file_name, doc in docs:
    print("Processing file:", file_name)
    result, usage = llm_structured(
        client=openai_client,
        user_prompt=json.dumps(doc),
        instructions=data_gen_instructions,
        output_type=Questions,
    )
    records.append({
        'filename': file_name,
        'questions': result.questions,
        'input_token_used': usage.input_tokens,
        'output_token_used': usage.output_tokens
    })
    print(f"\tProcessed {file_name}: {len(result.questions)} questions generated, \n\tinput tokens used: {usage.input_tokens}, \n\toutput tokens used: {usage.output_tokens}")

Processing file: id
	Processed id: 5 questions generated, 
	input tokens used: 168, 
	output tokens used: 83
Processing file: id
	Processed id: 5 questions generated, 
	input tokens used: 168, 
	output tokens used: 111
Processing file: id
	Processed id: 5 questions generated, 
	input tokens used: 168, 
	output tokens used: 94


What's the average number of input tokens across these 3 calls?

[X] 140

[ ] 1400

[ ] 14000

[ ] 140000

Ans: 168

In [8]:
import pandas as pd

ground_truth = pd.read_csv("./ground-truth.csv")
len(ground_truth)

360

## Searching the chunks

In [9]:
from gitsource import chunk_documents
from tqdm import tqdm
import numpy as np

from embedder import Embedder
embed = Embedder()

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

X = []
for i in tqdm(range(len(chunks))):
    chunk = chunks[i]
    batch_encode = embed.encode(chunk["content"])
    X.append(batch_encode)

X = np.array(X)

2026-07-13 14:13:27.151449440 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


295


100%|██████████| 295/295 [00:19<00:00, 15.17it/s]


### Building the search models

In [10]:
from minsearch import VectorSearch, Index

vector_search = VectorSearch(
    keyword_fields=["filename"]
)
vector_search.fit(X, chunks)

text_search = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

text_search.fit(chunks)

In [31]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


class HybridSearch:
    def __init__(self, k):
        self.k = k
    
    def search(self, query, num_results=10):
        text_results = text_search.search(query['question'], num_results=num_results)
        vector_results = vector_search.search(query['embed'], num_results=num_results)
        return rrf([text_results, vector_results], k=self.k, num_results=num_results)

### Q2. First result with text search

In [12]:
ground_truth = ground_truth.to_dict(orient='records')

In [13]:
q = ground_truth[0]["question"]
q_embed = embed.encode(q)

After running text_search for it, what's the filename of the first result?

In [14]:
r_ts = text_search.search(q, num_results=5)
r_ts[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

[ ] `01-agentic-rag/lessons/01-intro.md`

[X] `01-agentic-rag/lessons/03-rag.md`

[ ] `01-agentic-rag/lessons/13-function-calling.md`

[ ] `01-agentic-rag/lessons/10-rag-next-steps.md`

### Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

In [15]:
r_vs = vector_search.search(q_embed, num_results=5)
r_vs[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

[X] `01-agentic-rag/lessons/01-intro.md`

[ ] `01-agentic-rag/lessons/03-rag.md`

[ ] `04-evaluation/lessons/11-evaluation-intro.md`

[ ] `04-evaluation/lessons/12-rag-answers.md`

## Evaluation metrics

In [35]:
def calc_relevance_query(q, search_function):
    doc_id = q['filename']
    results = search_function.search(q['content'])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))
    
    return relevance

def calc_relevance_total(ground_truth, search_function):
    
    relevance_total = []

    for q in tqdm(ground_truth, desc="Queries", position=1, leave=False):
        relevance = calc_relevance_query(q, search_function)
        relevance_total.append(relevance)
    
    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

### Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

In [23]:
ts_ground_truth = [{
    'filename': gt['filename'],
    'content': gt['question']
                    } for gt in ground_truth]
ts_relevance = calc_relevance_total(ts_ground_truth, text_search)

ts_results = {
    'hit_rate': hit_rate(ts_relevance),
    'mrr': mrr(ts_relevance)
}

print(ts_results)


100%|██████████| 360/360 [00:00<00:00, 524.15it/s]

{'hit_rate': 0.8416666666666667, 'mrr': 0.6053858024691359}


What's the Hit Rate?

[ ] 0.55

[ ] 0.66

[ ] 0.76

[X] 0.88

### Q5. Evaluating vector search
Now evaluate `vector_search`


In [27]:
vs_ground_truth = [{
    'filename': gt['filename'],
    'content': embed.encode(gt['question'])
                    } for gt in ground_truth]
vs_relevance = calc_relevance_total(vs_ground_truth, vector_search)

vs_results = {
    'hit_rate': hit_rate(vs_relevance),
    'mrr': mrr(vs_relevance)
}

print(vs_results)

100%|██████████| 360/360 [00:00<00:00, 726.98it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.5646472663139328}


What's the MRR?

[ ] 0.35

[ ] 0.45

[X] 0.55

[ ] 0.65

### Q6. Tuning hybrid search

In [36]:
ks = [1, 50, 100, 200]
hs_ground_truth = [{
    'filename': gt['filename'],
    'content': {
        'question': gt['question'],
        'embed': embed.encode(gt['question'])
    }
                    } for gt in ground_truth]

for k in tqdm(ks, desc="k values", position=0):
    hs = HybridSearch(k=k)
    hs_k_relevance = calc_relevance_total(hs_ground_truth, hs)
    hs_results = {
        'hit_rate': hit_rate(hs_k_relevance),
        'mrr': mrr(hs_k_relevance)
    }
    print(f"Hybrid search results for k={k}: {hs_results}")

k values:  25%|██▌       | 1/4 [00:01<00:03,  1.05s/it]

Hybrid search results for k=1: {'hit_rate': 0.9083333333333333, 'mrr': 0.6576917989417991}


k values:  50%|█████     | 2/4 [00:01<00:01,  1.02it/s]

Hybrid search results for k=50: {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801}


k values:  75%|███████▌  | 3/4 [00:03<00:01,  1.08s/it]

Hybrid search results for k=100: {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801}


k values: 100%|██████████| 4/4 [00:04<00:00,  1.06s/it]

Hybrid search results for k=200: {'hit_rate': 0.9083333333333333, 'mrr': 0.6479872134038801}


Which `k` gives the best MRR?

[X] 1

[ ] 50

[ ] 100

[ ] 200